# Physics-Informed Neural Networks (PINNs)

> Data, physics, or both? A 1-D damped harmonic oscillator is the cleanest
> way to *see* what a PINN actually buys you over a plain neural network.

This tutorial follows the framing made famous by [Ben Moseley's PINN
blog post](https://benmoseley.blog/my-research/so-what-is-a-physics-informed-neural-network/):
pick a problem with a known exact solution, give yourself a few noisy
observations in a small time window, and watch three models behave very
differently when you ask them to predict the future:

1. **NN, data only** — learns the training points, then collapses.
2. **PINN, physics only** — knows the ODE but takes longer to converge.
3. **Hybrid: data + physics** — matches the data *and* extrapolates.

Once that lands, the bridge to inverse problems, differentiable simulation,
and the next step — **neural operators** — is one short paragraph.

## 1. The problem — damped harmonic oscillator

We solve

$$\quad m\,\ddot u + \mu\,\dot u + k\,u = 0,\qquad u(0) = 1,\ \dot u(0) = 0$$

with $\omega_0 = \sqrt{k/m} = 4\pi$ and damping ratio
$\zeta = \mu/(2\sqrt{m k}) = 0.05$ (lightly underdamped). The exact
solution is a decaying cosine.

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.optim as optim
import matplotlib.pyplot as plt
torch.manual_seed(0); np.random.seed(0)

OMEGA0, ZETA = 4 * np.pi, 0.05
def exact(t):
    omega_d = OMEGA0 * np.sqrt(1 - ZETA**2)
    phi = np.arctan(-ZETA * OMEGA0 / omega_d)
    return (1 / np.cos(phi)) * np.exp(-ZETA * OMEGA0 * t) * np.cos(omega_d * t + phi)

## 2. The 12 noisy observations

We see data only in `t ∈ [0, 0.4]` — the early, easy part of the
oscillator. **Everything past 0.4 is extrapolation.** That's where
this tutorial earns its keep.

In [ ]:
N_data, T_DATA, T_MAX = 12, 0.4, 1.0
t_data_np = np.linspace(0, T_DATA, N_data)
u_data_np = exact(t_data_np) + 0.02 * np.random.randn(N_data)
t_data = torch.tensor(t_data_np, dtype=torch.float32).view(-1, 1)
u_data = torch.tensor(u_data_np, dtype=torch.float32).view(-1, 1)

## 3. A small MLP — same architecture for all three runs

In [ ]:
class FCN(nn.Module):
    def __init__(self, hidden=64, depth=4):
        super().__init__()
        layers = [nn.Linear(1, hidden), nn.Tanh()]
        for _ in range(depth - 1):
            layers += [nn.Linear(hidden, hidden), nn.Tanh()]
        layers.append(nn.Linear(hidden, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, t): return self.net(t)

## 4. The physics residual via autograd

`u_t` and `u_tt` come from `torch.autograd.grad`. The trick is
`create_graph=True` on the first call so PyTorch builds the higher-order
graph needed for the second derivative.

In [ ]:
def physics_residual(model, t):
    t = t.requires_grad_(True)
    u   = model(t)
    u_t = torch.autograd.grad(u, t, torch.ones_like(u), create_graph=True)[0]
    u_tt = torch.autograd.grad(u_t, t, torch.ones_like(u_t), create_graph=True)[0]
    return u_tt + 2 * ZETA * OMEGA0 * u_t + OMEGA0**2 * u

t_phys = torch.linspace(0, T_MAX, 200).view(-1, 1)

## 5. Train all three models

The single `train` function takes `w_data` and `w_phys` weights.
Three different settings give three completely different behaviours.

| run | `w_data` | `w_phys` | what it sees |
|-----|----------|----------|--------------|
| (a) | 1 | 0 | data only — vanilla regression |
| (b) | 0 | 1e-3 | physics only — a PDE solver |
| (c) | 1 | 1e-3 | both — data and physics |

**Optimiser recipe**: Adam to find the basin of attraction, then a small
number of LBFGS steps to polish. Pure Adam plateaus around an MSE of
~0.1 on this problem; LBFGS drops it to ~5e-4. This Adam → LBFGS
pattern is the standard for vanilla PINNs.

In [ ]:
def total_loss(m, w_data, w_phys):
    loss = torch.zeros(())
    if w_data:
        loss = loss + w_data * ((m(t_data) - u_data)**2).mean()
    if w_phys:
        r = physics_residual(m, t_phys)
        loss = loss + w_phys * (r**2).mean()
        # Soft IC: u(0)=1, u_t(0)=0
        t0 = torch.zeros(1, 1, requires_grad=True)
        u0 = m(t0)
        u0_t = torch.autograd.grad(u0, t0, torch.ones_like(u0), create_graph=True)[0]
        loss = loss + 100 * ((u0 - 1)**2 + u0_t**2).mean()
    return loss

def train(w_data, w_phys, adam_epochs=3000, lbfgs_iters=40):
    m = FCN()
    opt = optim.Adam(m.parameters(), lr=2e-3)
    for ep in range(adam_epochs):
        opt.zero_grad(); total_loss(m, w_data, w_phys).backward(); opt.step()
    if w_phys:
        opt = optim.LBFGS(m.parameters(), lr=0.5, max_iter=20,
                          tolerance_grad=1e-7)
        def closure():
            opt.zero_grad(); L = total_loss(m, w_data, w_phys)
            L.backward(); return L
        for k in range(lbfgs_iters):
            opt.step(closure)
    return m

m_nn   = train(1.0, 0.0)
m_pinn = train(0.0, 1e-3)
m_hyb  = train(1.0, 1e-3)

## 6. Look at what each model learned

The hero plot. Yellow shading marks the data region — everything to the
right of it is extrapolation. The NN flatlines / drifts the moment data
runs out; the pure PINN gets the physics right but is sensitive to
initialisation; the hybrid does what we actually want.

In [ ]:
t_eval_np = np.linspace(0, T_MAX, 400)
t_eval = torch.tensor(t_eval_np, dtype=torch.float32).view(-1, 1)

def predict(m):
    with torch.no_grad():
        return m(t_eval).numpy().flatten()

u_true = exact(t_eval_np)
u_nn, u_pinn, u_hyb = predict(m_nn), predict(m_pinn), predict(m_hyb)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)
for ax, lab, u in zip(axes, ['NN', 'PINN', 'hybrid'], [u_nn, u_pinn, u_hyb]):
    ax.plot(t_eval_np, u_true, '--', color='gray', lw=1, label='exact')
    ax.plot(t_eval_np, u, color='#2ca02c', lw=2)
    ax.scatter(t_data_np, u_data_np, s=20, color='#1f77b4', zorder=5)
    ax.axvspan(0, T_DATA, color='#fff7e0', alpha=0.6)
    ax.axvline(T_DATA, color='#999', ls=':', lw=1)
    ax.set_xlabel('t'); ax.set_title(lab)
axes[0].set_ylabel('u(t)')
plt.tight_layout(); plt.show()

## 7. Why this matters — inverse problems

The moment your forward solver is **end-to-end differentiable**, the PDE
coefficients become *trainable parameters*. Given the same 12 noisy
observations, you can recover $\omega_0$ and $\zeta$ from data alone:

```python
omega0 = torch.nn.Parameter(torch.tensor(1.0))  # unknown!
zeta   = torch.nn.Parameter(torch.tensor(0.5))  # unknown!
opt = optim.Adam(list(m.parameters()) + [omega0, zeta], lr=1e-3)
```

This is the connection between standard PINN demos and the work I do on
[PhAST](https://cems-lab.github.io/PhAST/) — differentiable phase-field fracture,
where the public inverse benchmark recovers the material toughness $G_c$
from displacement observations. The math gets harder (non-convex energy, irreversibility,
operator-split stability) but **the autograd machinery is identical**.

## 8. The gotchas

- **Loss balancing.** `w_data = 1`, `w_phys = 1e-4`, `w_ic = 100` is a
  starting point, not a recipe. The right weights are problem-specific
  and recent work (NTK-PINN, self-adaptive PINNs) automates this.
- **Tanh, not ReLU.** ReLU's zero curvature kills second derivatives.
  Stick to Tanh, SiLU, or Gaussian activations for any PINN.
- **Stiff or chaotic systems.** PINNs notoriously fail here. See
  [Krishnapriyan 2021](https://arxiv.org/abs/2109.01050) for a clean
  diagnosis ("Characterizing possible failure modes in PINNs").

## 9. What comes next — neural operators

A PINN trains **one** network to solve **one** problem. Change $\omega_0$
or the initial condition and you retrain. **Neural operators** learn the
*solution map* — they take a PDE input field (forcing, geometry, IC) and
produce the solution field, for any input from the same family. Train once,
infer in milliseconds.

The two flagship families:

- [**DeepONet**](https://arxiv.org/abs/1910.03193) (Lu, Karniadakis, 2019) —
  branch + trunk decomposition.
- [**Fourier Neural Operator (FNO)**](https://arxiv.org/abs/2010.08895)
  (Li, Anandkumar et al., 2020) — learns in Fourier space.

Siddhartha Mishra (ETH Zürich) has a [3-lecture course at
CIRM](https://www.youtube.com/watch?v=5CnctvgyssU) that derives the theory
from scratch — recommended next watch.

## What you've built

- A PINN that solves an ODE with **no training data** beyond initial
  conditions.
- A hybrid model that matches data **and** extrapolates correctly outside
  the training window — the Moseley-style "magic" that PINNs are
  famous for.
- The autograd-residual pattern that generalises to any PDE.
- The vocabulary to read modern PINN literature, plus a pointer to neural
  operators (the next step beyond a single-PDE PINN).

**For the rest of the series**, see [tutorials](/tutorials/). For my own
research on differentiable phase-field fracture and inverse problems, see
the open [PhAST documentation](https://cems-lab.github.io/PhAST/),
[source](https://github.com/CEMS-Lab/PhAST), and
[preprint](https://arxiv.org/abs/2606.23458).